In [1]:
import pandas as pd
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [2]:
#Load Data and Create a Vector Store
from utils.data_loader import load_data
from utils.metrics import compute_metrics

df = load_data()

metrics = compute_metrics(df)

documents = [f"{k}: {v}" for k, v in metrics.items()]

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_texts(
    texts=documents,
    embedding=embeddings,
    collection_name="sales_data"
)

In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [4]:
llm = ChatOpenAI(temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a business intelligence analyst.

Use the context below to answer the question clearly and provide insights.

Context:
{context}

Question:
{question}
""")

def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
)

In [5]:
import subprocess
import sys
import ipywidgets as widgets
from IPython.display import display, HTML

def launch_streamlit(_):
    subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "app/streamlit_app.py"],
        creationflags=subprocess.CREATE_NEW_CONSOLE
    )

button = widgets.Button(description="🚀 Launch AI App")
button.on_click(launch_streamlit)

display(button)


Button(description='🚀 Launch AI App', style=ButtonStyle())